# Qiskit Fall Fest 2026, Challenge 1: Biggest GHZ state

Opens Mon, Nov 16, 2026 at 6:00 PM. Closes Sun, Nov 22, 2026 at 11:59 PM. Both times are Clemson time (US Eastern).

**The task.** Entangle the most qubits in a GHZ state on real hardware.

Start from Monday's notebook. Its Lab 3 built the GHZ circuit and sent it to a processor. This notebook adds what the challenge needs: a second circuit, one number that shows whether your qubits really are entangled, and the line of text that you turn in.

How to use this notebook:

- Run the three code cells of the Setup section first.
- Read "The task in detail". It says what counts and how a run is scored.
- Run the cells under "The tools". Their code is complete. They end with a practice run on a simulator, a program that imitates a quantum computer. It runs on the computer that runs this notebook.
- Do your own work under "Your attempt". A practice run needs no account. A run that counts needs a real quantum computer.
- Turn in the line that the notebook prints. The steps are under "Turn it in".

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## The task in detail

### The state

A GHZ state on $n$ qubits is an equal superposition of "all qubits are 0" and "all qubits are 1":

$$\lvert \text{GHZ}_n \rangle = \frac{\lvert 00 \ldots 0 \rangle + \lvert 11 \ldots 1 \rangle}{\sqrt{2}}$$

Its qubits are entangled: the state of the group cannot be described one qubit at a time. Monday's Lab 3 made it with an H gate on the first qubit and a chain of CX gates. The more qubits take part, the more gates the circuit has, and the more chances there are for an error. The challenge is to keep the state alive on as many qubits as you can.

### Why one set of counts is not enough

Measure a GHZ state and half of the shots give all zeros and the other half all ones. A shot is one run of a circuit, and the counts say how many shots gave each result. A machine that tossed an ordinary coin and then wrote all zeros or all ones would produce the same counts with no entanglement at all. What sets the GHZ state apart is the plus sign between its two parts, and a measurement of 0s and 1s cannot see that sign.

A second circuit can. It is the same GHZ circuit followed by an H gate on every qubit, and then the measurement. For a true GHZ state every result of this second run has an even number of 1s. For the coin-toss machine the number of 1s is even only half of the time.

### Two runs, three counts, one number

- **The Z run** is the GHZ circuit followed by a measurement of every qubit. From it you take `zeros`, the number of shots that are all zeros, and `ones`, the number of shots that are all ones.
- **The X run** is the same GHZ circuit, then an H gate on every qubit, then the measurement. From it you take `even`, the number of shots with an even number of 1s.

Each run has 4,096 shots. The three counts go into one number:

$$B = \frac{\text{zeros} + \text{ones} + \text{even}}{4{,}096} - 1$$

A perfect GHZ state gives $B = 1$. A state in which the $n$ qubits are not all entangled with each other gives 0.5 or less on average. The reason is a fact about the fidelity, a number between 0 and 1 that says how close the state you made is to the ideal one, with 1 meaning identical. $B$ is never larger than the fidelity, and a fidelity above one half is possible only if all $n$ qubits are entangled together. So a large enough $B$ is a certificate: it proves the entanglement from counts alone.

### The rules

- A real quantum computer. The first choice is an IBM processor through your own IBM Quantum account. The second option is a device reached through qBraid. A simulator does not count.
- One job with two circuits, the Z run and the X run, with 4,096 shots each, on the same physical qubits, the numbered qubits on the chip. A job is one request to run circuits. On the second option the two circuits go as two jobs, and the device places them on its own qubits. The line then carries both job IDs.
- Each circuit is gates followed by one final measurement of all $n$ qubits. Nothing is measured before the end, no qubit is put back to 0 on the way, and no gate depends on a measurement result.
- The options of the Sampler stay as they are. The Sampler is the Qiskit tool that runs circuits and returns their counts. Every shot counts.
- The job is created after the challenge opens and before it closes. The submission form closes at the same time, and a line counts only if the form has it by Sun, Nov 22 at 11:59 PM, so leave time for the queue.

### When a run is valid

A run is valid when $n$ is at least 2 and all three of these hold.

1. `zeros` and `ones` are each at least 410 of the 4,096 shots. A GHZ state gives both. Qubits that were never touched give only zeros.
2. No single result of the X run has more than $30 + 8{,}192 / 2^{n-1}$ shots, rounded down. That is 46 for $n = 10$. The X run of a GHZ state spreads over $2^{n-1}$ different results. A run without the H gates piles up on two.
3. $B \geq 0.55$, which is the same as `zeros` + `ones` + `even` being at least 6,349.

The mark is 0.55 and not 0.5 because a number computed from 4,096 shots scatters by about 0.01 from one run to the next. The extra 0.05 keeps a state without entanglement from passing by luck.

### Your score

A valid run on $n$ qubits scores 10 points for each qubit, up to 100 points at 10 qubits or more. A run that is not valid scores 0. You may turn in a line as often as you like, and your best valid line counts. How the event uses the scores is explained on the event page.

### What to expect

A real processor is much harder on this state than the simulator in this notebook. On Nov 14, 2026, the settings this notebook starts with (Monday's chain on the first $n$ qubits of the chip, optimization level 1; the optimization level, from 0 to 3, sets how hard Qiskit works to shorten the circuit for the chip) gave $B = 0.64$ at 8 qubits and $B = 0.39$ at 12 qubits on the IBM processor ibm_fez. The simulated processor gives 0.72 at 12 qubits with the same settings. So use the practice run to test your code, and do not read a score from it. Ten qubits take at least one good decision of your own.

## The tools

The code in this part is complete. Run each cell and read what it does. You will call these functions in "Your attempt".

The first cell builds the two circuits.

- `ghz_body(n)` is Monday's `ghz(n)` without its last line, the measurement. The notebook adds the measurements itself, because the X run needs its H gates in front of them.
- `build_pair(body, backend, qubits, level, seed)` transpiles the GHZ circuit and returns the Z circuit, the X circuit and the list of physical qubits. To transpile is to rewrite a circuit in the gates of one processor and place it on that processor's qubits. A backend is the machine, real or simulated, that runs a circuit. The physical qubits are the numbered qubits on the chip. With `qubits=None` the transpiler chooses them. With a list, entry $k$ is the physical qubit for qubit $k$ of your circuit. `level` is the optimization level, from 0 to 3: how hard the transpiler works to shorten the circuit. `seed` fixes the transpiler's random choices.
- Both circuits are made from one transpiled copy of the GHZ circuit, so they sit on the same physical qubits. In the X circuit the H gate is written as `rz`, `sx`, `rz`, three gates that every IBM processor has.

In [ ]:
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager

SHOTS = 4096          # shots for each of the two circuits. The rules fix this number.
TARGET = 10           # the number of qubits that earns the full score
PASS_MARK = 0.55      # the smallest B that counts


def ghz_body(n):
    """Monday's GHZ circuit without its last line, the measurement."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    return qc


def checked_body(body):
    """Return the body, or stop with a clear message if it measures."""
    if any(inst.operation.name == "measure" for inst in body.data):
        raise ValueError("my_body must return a circuit without measurements. Take out measure_all() or measure().")
    return body


def build_pair(body, backend, qubits=None, level=1, seed=None):
    """Transpile a GHZ circuit. Return the Z circuit, the X circuit and the physical qubits they use."""
    n = checked_body(body).num_qubits
    pm = generate_preset_pass_manager(optimization_level=level, backend=backend,
                                      initial_layout=qubits, seed_transpiler=seed)
    isa = pm.run(body)
    where = isa.layout.final_index_layout()      # the physical qubit that holds GHZ qubit k at the end
    pair = []
    for with_h in (False, True):
        circuit = isa.copy()
        meas = ClassicalRegister(n, "meas")      # n ordinary bits for the results
        circuit.add_register(meas)
        if with_h:
            for q in where:                      # an H gate, written in the processor's own gates
                circuit.rz(np.pi / 2, q)
                circuit.sx(q)
                circuit.rz(np.pi / 2, q)
        circuit.measure(where, meas)
        pair.append(circuit)
    return pair[0], pair[1], [int(q) for q in where]

The second cell is the self-check, `replay`. It works out, without noise and without a quantum computer, what a circuit does, and says whether it is what the challenge asks for: exactly $n$ measurements, all at the end, and before them the GHZ state (for the Z run) or the GHZ state followed by an H gate on every qubit (for the X run). Run it on your circuits before you spend time on a real processor.

How it decides: a GHZ state is the only state for which a certain list of $n$ yes-or-no questions all have the answer yes. `replay` asks each of them. You do not need to follow the code to use it.

In [ ]:
from qiskit.quantum_info import Pauli, StabilizerState, Statevector


def replay(circuit, n, x_run):
    """Return (True, "") if the circuit is this challenge's Z run (or X run if x_run is True), else (False, why)."""
    ops = list(circuit.data)
    if any(inst.operation.name in ("reset", "if_else", "while_loop", "for_loop", "switch_case") for inst in ops):
        return False, "the circuit has a reset or a gate that depends on a measurement"
    measures = [inst for inst in ops if inst.operation.name == "measure"]
    if len(measures) != n or any(inst.operation.name != "measure" for inst in ops[len(ops) - len(measures):]):
        return False, "the circuit needs exactly " + str(n) + " measurements, all at the end"
    measures.sort(key=lambda inst: circuit.find_bit(inst.clbits[0]).index)
    measured = [circuit.find_bit(inst.qubits[0]).index for inst in measures]
    if len(set(measured)) != n:
        return False, "every qubit must be measured once"
    touched = sorted({circuit.find_bit(q).index for inst in ops for q in inst.qubits})
    place = {q: k for k, q in enumerate(touched)}
    body = QuantumCircuit(len(touched))          # the same gates on a small register, without the measurements
    for inst in ops:
        if inst.operation.name not in ("measure", "barrier", "delay"):
            body.append(inst.operation, [place[circuit.find_bit(q).index] for q in inst.qubits])
    try:
        state = StabilizerState(body)            # fast, for circuits made of H, CX, CZ and quarter turns
    except Exception:
        if len(touched) > 20:
            return False, "this circuit is too large to replay on a laptop: use optimization level 1, or at most 20 qubits"
        state = Statevector(body)                # any other circuit, up to 20 qubits
    spots = [place[q] for q in measured]
    first, second = ("Z", "X") if x_run else ("X", "Z")
    questions = [(first * n, spots)] + [(second * 2, [spots[k], spots[k + 1]]) for k in range(n - 1)]
    questions += [("Z", [place[q]]) for q in touched if q not in measured]     # every other qubit must end in 0
    for label, qubits in questions:
        if abs(state.expectation_value(Pauli(label), qubits) - 1) > 1e-6:
            return False, "this is not the GHZ state" + (" followed by an H gate on every qubit" if x_run else "")
    return True, ""

The third cell turns counts into the numbers of the challenge.

- `numbers(counts_z, counts_x, n)` returns `zeros`, `ones`, `even` and `xmax`, the number of shots of the most frequent result of the X run.
- `certificate(zeros, ones, even)` is $B$.
- `problems(n, zeros, ones, even, xmax)` lists the reasons why a run is not valid. An empty list means that it is valid.
- `submission_line(...)` writes the line you turn in.

In [ ]:
import qiskit
import qiskit_ibm_runtime


def numbers(counts_z, counts_x, n):
    """Return zeros, ones, even and xmax from the counts of the Z run and of the X run."""
    zeros = counts_z.get("0" * n, 0)
    ones = counts_z.get("1" * n, 0)
    even = sum(shots for bits, shots in counts_x.items() if bits.count("1") % 2 == 0)
    xmax = max(counts_x.values())
    return zeros, ones, even, xmax


def certificate(zeros, ones, even):
    """The number B of this challenge."""
    return (zeros + ones + even) / SHOTS - 1


def problems(n, zeros, ones, even, xmax):
    """Return the list of reasons why a run is not valid. An empty list means the run is valid."""
    found = []
    if n < 2:
        found.append("n must be at least 2: one qubit cannot be entangled with anything")
    if zeros < 410 or ones < 410:
        found.append("zeros and ones must each be at least 410 shots")
    limit = 30 + 8192 // 2 ** (n - 1)
    if xmax > limit:
        found.append("one result of the X run has " + str(xmax) + " shots; the limit for n = " + str(n) + " is " + str(limit))
    if zeros + ones + even < 6349:
        found.append("B is below " + str(PASS_MARK))
    return found


def submission_line(n, device, job_id, zeros, ones, even, xmax, created_utc, second_option=None):
    """The one line of text that you turn in."""
    versions = "qiskit=" + qiskit.__version__ + " "
    versions += ("qbraid=" + second_option) if second_option else ("runtime=" + qiskit_ibm_runtime.__version__)
    return ("QFF26-C1 n=" + str(n) + " device=" + str(device) + " job=" + str(job_id) + " shots=" + str(SHOTS)
            + " zeros=" + str(zeros) + " ones=" + str(ones) + " even=" + str(even) + " xmax=" + str(xmax)
            + " B=" + format(certificate(zeros, ones, even), ".3f") + " utc=" + created_utc + " " + versions)

The last cell of this part is a practice run with the settings this notebook starts with: Monday's chain of 6 qubits on the first six qubits of the chip, optimization level 1. It always runs on the simulated processor, whatever the switches in the Setup section say, so it needs no account and sends nothing. It takes a few seconds.

Read its output from top to bottom: the physical qubits, the result of the self-check for both circuits, the three counts with `xmax`, $B$, and whether a run with these numbers would be valid.

In [ ]:
from qiskit_ibm_runtime import SamplerV2

practice_n = 6
z_circuit, x_circuit, where = build_pair(ghz_body(practice_n), SIMULATED_BACKEND,
                                         qubits=list(range(practice_n)), level=1, seed=7)
print("physical qubits:", where)
print("self-check, Z run:", replay(z_circuit, practice_n, False)[0], "| X run:", replay(x_circuit, practice_n, True)[0])

practice = SamplerV2(mode=SIMULATED_BACKEND).run([z_circuit, x_circuit], shots=SHOTS).result()
zeros, ones, even, xmax = numbers(practice[0].data.meas.get_counts(), practice[1].data.meas.get_counts(), practice_n)
print("zeros:", zeros, "| ones:", ones, "| even:", even, "| xmax:", xmax)
print("B =", round(certificate(zeros, ones, even), 3))
print("not valid because:" if problems(practice_n, zeros, ones, even, xmax) else "these numbers would be valid",
      *problems(practice_n, zeros, ones, even, xmax))
print("This was a practice run on the simulated processor. It is not a submission.")

On the simulated processor $B$ comes out near 0.86 for 6 qubits. `zeros` and `ones` are each a little under half of the 4,096 shots, `even` is well above half, and `xmax` stays below its limit of 286 for $n = 6$.

## Your attempt

From here on the work is yours. Run the cells of this part in order.

The first cell chooses the processor. With `USE_HARDWARE = False` it is the simulated one. With `USE_HARDWARE = True` and `HARDWARE = "ibm"` it is the least busy real one. Looking at a processor costs no quantum time, which is the time a real processor spends on your jobs. The cells below use this processor, so the qubits you look at are the qubits you run on. If you want one particular processor, replace the `least_busy` line by `backend = service.backend("ibm_fez")`, with the name you want. After you change a switch in the Setup section, run the setup cells and then this cell again.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
service = connect("ibm")
backend = service.least_busy(operational=True, simulator=False)
on_simulator = type(service).__name__ == "SimulatedService"      # True: the simulated processor, and nothing is sent
print("backend:", backend.name, "with", backend.num_qubits, "qubits")

The next cell holds your choices. Change them, then run the cells after it again.

- **How many qubits.** `n`. More qubits score more, as long as the run stays valid.
- **Which physical qubits.** `my_qubits`. Qubits on one chip differ a lot in quality, and so do the connections between them. The default, the first `n` qubits in a row, is rarely the best choice, and it is a chain only up to 16 qubits. You can pick a list by hand: the cell after this one prints what the processor reports about the qubits in your list. Or set `my_qubits = None` and let the transpiler choose, which it does with more care at a higher optimization level.
- **The shape of the circuit.** `my_body`. Monday's chain starts at one end, so the first qubit waits while all the others are added. Any circuit that prepares the GHZ state is allowed. One that grows from the middle is half as deep.
- **When to spend real quantum time.** The Open Plan of IBM Quantum, the plan you do not pay for, gives up to 10 minutes of quantum time in every 28-day window (IBM Quantum documentation, "Plans overview", read on Nov 13, 2026). On Nov 14, 2026 one job of sixteen GHZ circuits of 4,096 shots each used 19 seconds, and on Nov 16 one attempt, a job of two such circuits, used 4 seconds.

If your list is a chain of neighbors on the chip, the optimization level and the seed change nothing that matters for this circuit. If it is not a chain, the transpiler adds gates to bring qubits together. Then the level and the seed matter, the physical qubits that the run prints can come out in another order than your list, and a practice run with many qubits can take minutes.

In [ ]:
n = 6                      # how many qubits
my_qubits = list(range(n))  # the physical qubits, or None to let the transpiler choose
my_level = 1               # optimization level, 0 to 3
my_seed = 7                # fixes the transpiler's random choices


def my_body(n):
    """Your GHZ circuit, without measurements. Change the shape here if you like."""
    return ghz_body(n)

The next cell prints what the processor reports about the qubits in `my_qubits`: the readout error of each qubit, the error of its one-qubit gates, and the error of the two-qubit gate between each qubit and the next one in your list. A reported error is the processor's own estimate, from its last calibration, of how often an operation goes wrong. The readout error is the chance that a measurement returns the wrong bit. Smaller is better.

Monday's chain uses exactly the pairs that the cell prints. A pair marked "not neighbors" has no connection on the chip. Reading these numbers costs no quantum time, so you can change `my_qubits` in the cell above, run it, and run this cell again as often as you like.

In [ ]:
record = backend.target                      # what the processor reports about its qubits and gates
two_qubit_gate = next(name for name in ("cz", "ecr", "cx") if name in record.operation_names)


def reported(props, digits):
    """A reported error as text."""
    return "not reported" if props is None or props.error is None else format(props.error, "." + str(digits) + "f")


if my_qubits is None:
    print("my_qubits is None: the transpiler will choose the qubits, so there is no list to look at.")
elif not all(isinstance(q, (int, np.integer)) and 0 <= q < backend.num_qubits for q in my_qubits):
    print("my_qubits must hold whole numbers from 0 to", backend.num_qubits - 1)
elif len(set(my_qubits)) != len(my_qubits):
    print("my_qubits holds a qubit more than once. Each qubit can appear only once.")
else:
    for q in my_qubits:
        print("qubit", q, "| readout error:", reported(record["measure"].get((q,)), 4),
              "| one-qubit gate error:", reported(record["sx"].get((q,)), 5))
    for a, b in zip(my_qubits, my_qubits[1:]):
        link = record[two_qubit_gate].get((a, b)) or record[two_qubit_gate].get((b, a))
        print("pair", a, b, "| two-qubit gate error:", reported(link, 4) if link else "not neighbors")

The next cell builds your two circuits for the processor, checks them with `replay`, and runs them. On the simulated processor it is a practice run. On a real processor it sends one job with both circuits, and that run can count.

If the self-check fails, the cell stops before anything is sent and says why.

**If your job is stuck in the queue.** Interrupt the kernel (Jupyter and qBraid Lab: Kernel menu, Interrupt Kernel. Google Colab: Runtime menu, Interrupt execution). Interrupting stops the waiting, not the job: the job stays in IBM's queue. When its turn comes it still runs and uses quantum time, and a challenge notebook has no stored results to fall back on. So cancel it before you run the cell again: run `job.cancel()` in a new cell. The Workloads page of your IBM Quantum account (quantum.cloud.ibm.com) lists your jobs and shows which ones still wait.

**Hardware cell (IBM Quantum).** This cell runs on the processor that the first cell of this part chose. If that is a real processor, the cell uses your own IBM Quantum account and your quantum time. If it is the simulated one, the cell uses no account. If a switch in the Setup section was changed since the processor was chosen, the cell stops before anything is sent and asks you to run the first cell of this part again.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2

assert on_simulator != (USE_HARDWARE and str(HARDWARE).strip().lower() == "ibm"), "A switch in the Setup section was changed after the processor was chosen. Run the first cell of this part again."
assert n >= 2, "n must be at least 2: one qubit cannot be entangled with anything."
z_circuit, x_circuit, where = build_pair(my_body(n), backend, my_qubits, my_level, my_seed)
print("backend:", backend.name, "| physical qubits:", where)

ok_z, why_z = replay(z_circuit, n, False)
ok_x, why_x = replay(x_circuit, n, True)
assert ok_z, "The Z circuit did not pass the self-check: " + why_z
assert ok_x, "The X circuit did not pass the self-check: " + why_x

mode = backend
if on_simulator and n > 14:                  # simulated processor: a method that copes with many qubits
    mode = AerSimulator.from_backend(backend, method="matrix_product_state")
job = SamplerV2(mode=mode).run([z_circuit, x_circuit], shots=SHOTS)
if not on_simulator:
    print("job sent to", backend.name, "| job ID:", job.job_id())
result = job.result()
counts_z = result[0].data.meas.get_counts()
counts_x = result[1].data.meas.get_counts()
run_n, run_device, run_practice, run_job, run_counts = n, backend.name, on_simulator, job, (counts_z, counts_x)      # what this job ran: the next cell uses these
print("both circuits ran:", sum(counts_z.values()), "and", sum(counts_x.values()), "shots")

The next cell works out your numbers and prints the line to turn in. It prints the line only for a valid run on a real processor. It takes the number of qubits and the device from the run itself, so changing your choices afterwards does not change the line.

In [ ]:
from datetime import timezone

zeros, ones, even, xmax = numbers(*run_counts, run_n)
print("n:", run_n, "| zeros:", zeros, "| ones:", ones, "| even:", even, "| xmax:", xmax)
print("B =", round(certificate(zeros, ones, even), 3), "| pass mark:", PASS_MARK)
trouble = problems(run_n, zeros, ones, even, xmax)
if run_practice:
    print("This was a practice run on the simulated processor. It is not a submission.")
    print("With these numbers a real run would" + (" not be valid: " + "; ".join(trouble) if trouble else " be valid."))
elif trouble:
    print("This run is not valid: " + "; ".join(trouble))
    print("Try fewer qubits, better qubits or a shorter circuit.")
else:
    created = run_job.creation_date.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("Valid run on", run_n, "qubits. Score:", min(100, (200 * run_n + TARGET) // (2 * TARGET)), "points. Turn in this line:")
    print(submission_line(run_n, run_device, run_job.job_id(), zeros, ones, even, xmax, created))

### The second option

The cell below sends the same two runs to another quantum computer through qBraid. A circuit transpiled for an IBM processor does not fit a different machine, so the cell sends your circuit before transpiling, and the device places it on its own qubits. You cannot choose physical qubits on this route.

How to use the cell:

1. In the Setup section set `USE_HARDWARE = True` and `HARDWARE = "qbraid"`, then run the first two setup cells again. The first time, the setup cell installs the `qbraid` package and asks for a restart.
2. Run the cell below. It prints one line for each device, with its id, its status, its number of qubits and whether it is a simulator. Then it stops with an error saying that the device `PASTE_DEVICE_ID` was not found. That stop is expected on the first run, because no device is chosen yet. Copy the id of a device that is online, is a real quantum computer and has at least `n` qubits into `QBRAID_DEVICE` in the Setup section, run that cell, then run the cell below again.
3. The cell checks your circuit with `replay`, sends the Z run and the X run as two jobs, waits for both, and prints your numbers and, for a valid run, the line to turn in. The line then carries both job IDs and the time at which the jobs were sent.

Four lines in the cell, from `formats = ...` to `set_target_program_type`, ask the device for OpenQASM 3, the newer of two text formats in which a circuit can travel. On the tested versions the `qbraid` package can only load that one.

`wait_for_final_state()` also returns when a job has failed, and reading the counts then stops with an error. A job that is put on hold never ends the wait. In that case interrupt the kernel and ask an organizer.

This cell was checked against the `qbraid` 0.13.0 package with the network replaced by a stand-in. It has not been run with a real qBraid account. Ask an organizer whether the second option has been confirmed for this challenge before you rely on it.

**Hardware cell, second option (qBraid).** With `USE_HARDWARE = True` and `HARDWARE = "qbraid"` this cell uses your qBraid account. With any other setting it is skipped and uses no account.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version

provider = connect("qbraid")
if provider:
    for dev in provider.get_devices():      # one line for each device
        print(dev.id, dev.status(), "| qubits:", dev.num_qubits, "| simulator:", bool(dev.profile.get("simulator")))

    device = provider.get_device(QBRAID_DEVICE)
    formats = device.profile.program_spec   # the text formats this device accepts
    formats = formats if isinstance(formats, list) else [formats]
    if "qasm3" in [item.alias for item in formats]:
        device.set_target_program_type("qasm3")

    z_plain = checked_body(my_body(n))      # your circuit before transpiling, then measure
    z_plain.measure_all()
    x_plain = checked_body(my_body(n))      # the same circuit, an H gate on every qubit, then measure
    x_plain.h(range(n))
    x_plain.measure_all()
    ok_z, why_z = replay(z_plain, n, False)
    ok_x, why_x = replay(x_plain, n, True)
    assert ok_z, "The Z circuit did not pass the self-check: " + why_z
    assert ok_x, "The X circuit did not pass the self-check: " + why_x

    sent_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")     # the time the jobs are created
    job_z = device.run(z_plain, shots=SHOTS)
    job_x = device.run(x_plain, shots=SHOTS)
    job_z.wait_for_final_state()
    job_x.wait_for_final_state()
    counts_z = job_z.result().data.get_counts()
    counts_x = job_x.result().data.get_counts()

    zeros, ones, even, xmax = numbers(counts_z, counts_x, n)
    print("n:", n, "| zeros:", zeros, "| ones:", ones, "| even:", even, "| xmax:", xmax)
    print("B =", round(certificate(zeros, ones, even), 3), "| pass mark:", PASS_MARK)
    trouble = problems(n, zeros, ones, even, xmax)
    if sum(counts_z.values()) != SHOTS or sum(counts_x.values()) != SHOTS:
        trouble.append("each run needs exactly " + str(SHOTS) + " shots")
    if device.profile.get("simulator"):
        print("This device is a simulator. A run on it is not a submission.")
    elif trouble:
        print("This run is not valid: " + "; ".join(trouble))
    else:
        print("Valid run on", n, "qubits. Score:", min(100, (200 * n + TARGET) // (2 * TARGET)), "points. Turn in this line:")
        print(submission_line(n, QBRAID_DEVICE, str(job_z.id) + "+" + str(job_x.id), zeros, ones, even, xmax,
                              sent_at, second_option=version("qbraid")))

## Turn it in

1. Copy the whole line that starts with `QFF26-C1`. Do not change anything in it.
2. Open the submission form: [https://forms.gle/GVYREiQnWGFF18Sy9](https://forms.gle/GVYREiQnWGFF18Sy9)
3. Enter the email you registered with and a leaderboard name, paste the line, and send. The name is shown on the leaderboards of the event page. Type the same name every time: the boards keep the name you typed first in any of the five forms, and only an organizer can change it.

You can turn in a new line whenever you have a better one. Your best valid line counts.

## If something breaks

| What you see | What to do |
|---|---|
| The setup cell stops and asks for a restart | Restart the kernel, then run the notebook again from the top. It installed the tested packages |
| "The Z circuit did not pass the self-check" | Your `my_body` does not prepare the GHZ state. The message says what the check found |
| "my_body must return a circuit without measurements" | Take `measure_all()` or `measure()` out of `my_body`. The notebook adds the measurements itself |
| "too large to replay on a laptop" | Transpile at optimization level 1, or use at most 20 qubits |
| `NameError: name '...' is not defined` | Run the cells in order, from the top. The cells of "Your attempt" need the cells of "The tools" |
| An error from the transpiler about the layout | `my_qubits` needs exactly `n` different qubit numbers that exist on the backend |
| The run is valid on the simulated processor and not on a real one | Expected. A real processor has more errors. Try fewer qubits, better qubits or a shorter circuit |
| "one result of the X run has ... shots" | The X run did not spread out as a GHZ state does. Check that you did not change `build_pair` |
| The job stays queued | Interrupt the kernel and cancel the job: run `job.cancel()` in a new cell. Then run the cell that sends the job again later. A job that is not cancelled still runs and uses quantum time |
| The notebook asks for an API key | No IBM Quantum account is saved on this computer. Monday's notebook, section "Accounts", shows how to get the key |

## Where to go from here

- Every challenge stays open until Sun, Nov 22 at 11:59 PM, so you can come back to this one after the later sessions. Friday's session is about the errors that limit you here.
- Stuck? Ask in Discord between sessions.